In [9]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os
np.random.seed(42)
random.seed(42)

N_USERS = 5000
N_ORDERS = 25000

cities = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata', 'Ahmedabad']
cuisines = ['North Indian', 'South Indian', 'Chinese', 'Pizza', 'Biryani', 'Burgers', 'Healthy', 'Desserts']
payment_methods = ['UPI', 'Credit Card', 'Debit Card', 'Cash on Delivery', 'Wallet']
device_types = ['Android', 'iOS', 'Web']
promo_types = ['NONE', 'FLAT50', 'BOGO', 'FREEDELIVERY', 'NEWUSER']


In [3]:
#users base
users = pd.DataFrame({
    'user_id': [f'U{str(i).zfill(5)}' for i in range(1, N_USERS+1)],
    'city': np.random.choice(cities, N_USERS, p=[0.20, 0.18, 0.17, 0.12, 0.10, 0.09, 0.08, 0.06]),
    'age': np.random.normal(28, 7, N_USERS).clip(18, 55).astype(int),
    'device': np.random.choice(device_types, N_USERS, p=[0.55, 0.30, 0.15]),
    'signup_date': [datetime(2023, 1, 1) + timedelta(days=np.random.randint(0, 365)) for _ in range(N_USERS)],
    'is_premium': np.random.choice([0, 1], N_USERS, p=[0.75, 0.25]),
})


In [4]:
# Orders
order_user_ids = np.random.choice(users['user_id'], N_ORDERS, p=None)

In [5]:
# Simulate power users (20% users make 60% orders)
power_users = users['user_id'].sample(frac=0.20).values
order_user_ids = np.concatenate([
    np.random.choice(power_users, int(N_ORDERS * 0.60)),
    np.random.choice(users['user_id'], int(N_ORDERS * 0.40))
])
np.random.shuffle(order_user_ids)
order_user_ids = order_user_ids[:N_ORDERS]

start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 12, 31)

orders = pd.DataFrame({
    'order_id': [f'ORD{str(i).zfill(6)}' for i in range(1, N_ORDERS+1)],
    'user_id': order_user_ids,
    'order_date': [start_date + timedelta(days=np.random.randint(0, 365),
                                          hours=np.random.randint(0, 23),
                                          minutes=np.random.randint(0, 59)) for _ in range(N_ORDERS)],
    'cuisine': np.random.choice(cuisines, N_ORDERS, p=[0.22, 0.15, 0.14, 0.12, 0.14, 0.10, 0.07, 0.06]),
    'order_value': np.random.normal(320, 120, N_ORDERS).clip(80, 900).round(2),
    'delivery_time_mins': np.random.normal(32, 10, N_ORDERS).clip(10, 75).astype(int),
    'delivery_fee': np.random.choice([0, 20, 30, 40, 49], N_ORDERS, p=[0.30, 0.20, 0.25, 0.15, 0.10]),
    'payment_method': np.random.choice(payment_methods, N_ORDERS, p=[0.42, 0.22, 0.15, 0.12, 0.09]),
    'promo_used': np.random.choice(promo_types, N_ORDERS, p=[0.55, 0.12, 0.08, 0.15, 0.10]),
    'rating': np.random.choice([1, 2, 3, 4, 5], N_ORDERS, p=[0.03, 0.05, 0.12, 0.35, 0.45]),
    'cancelled': np.random.choice([0, 1], N_ORDERS, p=[0.92, 0.08]),
    'late_delivery': np.random.choice([0, 1], N_ORDERS, p=[0.85, 0.15]),
})

In [6]:
#funnel events
funnel_stages = ['app_open', 'search', 'restaurant_view', 'add_to_cart', 'checkout', 'order_placed']
funnel_rates = [1.0, 0.72, 0.58, 0.41, 0.28, 0.21]

N_SESSIONS = 50000
funnel_data = []
for i in range(N_SESSIONS):
    user = np.random.choice(users['user_id'])
    session_date = start_date + timedelta(days=np.random.randint(0, 365))
    reached = 'app_open'
    for stage, rate in zip(funnel_stages, funnel_rates):
        if np.random.random() < rate:
            reached = stage
            funnel_data.append({
                'session_id': f'S{str(i).zfill(6)}',
                'user_id': user,
                'stage': stage,
                'date': session_date,
                'device': np.random.choice(device_types, p=[0.55, 0.30, 0.15])
            })
        else:
            break

funnel_df = pd.DataFrame(funnel_data)

In [10]:
orders = orders.merge(users[['user_id', 'city', 'age', 'device', 'signup_date', 'is_premium']], on='user_id')

orders.to_csv('../data/orders.csv', index=False)
users.to_csv('../data/users.csv', index=False)
funnel_df.to_csv('../data/funnel_events.csv', index=False)

print(f"✅ Orders:        {len(orders):,}")
print(f"✅ Users:         {len(users):,}")
print(f"✅ Funnel events: {len(funnel_df):,}")
print("✅ All data saved to ../data/")


✅ Orders:        25,000
✅ Users:         5,000
✅ Funnel events: 118,656
✅ All data saved to ../data/
